# BloodBridge AI: Real Public Blood-Bank Directory Collection

This notebook focuses on gathering directory metadata from Esri India's Blood Bank Directory (historical directory attributed to MoHFW / data.gov.in). 

It only collects public reference metadata such as blood bank coordinates, cities, and states. This reference catalog is then utilized as metadata context for our synthetic operational datasets.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlencode
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Constants for data endpoints
ENDPOINT = 'https://livingatlas.esri.in/server1/rest/services/Health/IN_BloodBankDirectory_2017/MapServer/0/query'
SOURCE = 'Esri India Blood Bank Directory 2017 (MoHFW/data.gov.in)'

### Helper Function to Download Catalog Records

We implement a paginated download logic to pull feature records from the Esri map server REST API. The function will run page by page until all elements are collected or the requested limit is reached.

In [2]:
def download_records(endpoint: str = ENDPOINT, limit: int = 0) -> list[dict]:
    """Helper to pull location catalog features page-by-page."""
    records = []
    offset = 0
    page_size = 1000
    
    headers = {
        'User-Agent': 'BloodBridgeAI academic research project (contact@example.invalid)'
    }
    
    while True:
        parameters = {
            'where': '1=1',
            'outFields': 'objectid,blood_bank_name,state,district,city,latitude,longitude',
            'returnGeometry': 'false',
            'f': 'json',
            'resultOffset': offset,
            'resultRecordCount': page_size
        }
        
        url = f"{endpoint}?{urlencode(parameters)}"
        response = requests.get(url, headers=headers, timeout=30)
        
        # Handle non-JSON server error pages
        if 'html' in response.headers.get('content-type', '').lower():
            soup = BeautifulSoup(response.text, 'html.parser')
            error_text = soup.get_text(' ', strip=True)[:200]
            raise RuntimeError(f"API Server returned HTML error: {error_text}")
            
        response.raise_for_status()
        features = response.json().get('features', [])
        if not features:
            break
            
        records.extend(item['attributes'] for item in features)
        
        if limit and len(records) >= limit:
            return records[:limit]
            
        if len(features) < page_size:
            break
            
        offset += page_size
        
    return records

### Executing Catalog Collection

Let's fetch the data, structure it as a Pandas DataFrame, and persist it to the `data/` directory for subsequent pipeline steps.

In [3]:
# Run download
raw_records = download_records()  # Use download_records(limit=100) for local testing

df = pd.DataFrame(raw_records)
df = df.rename(columns={
    'objectid': 'source_object_id',
    'blood_bank_name': 'bank_name'
})

# Insert custom identifiers and tracking fields
df.insert(0, 'bank_id', 'official_' + df['source_object_id'].astype(str))
df['data_source'] = SOURCE
df['collected_at_utc'] = datetime.now(timezone.utc).isoformat()
df['directory_snapshot'] = '2017 (historical; not live availability)'

output_cols = [
    'bank_id', 'bank_name', 'city', 'district', 'state',
    'latitude', 'longitude', 'data_source', 'collected_at_utc', 'directory_snapshot'
]

output_file = Path('data') / 'bloodbank_reference_catalog.csv'
output_file.parent.mkdir(parents=True, exist_ok=True)

# Save catalog to CSV
df.reindex(columns=output_cols).to_csv(output_file, index=False)
print(f"Saved {len(df):,} real public directory records to {output_file.resolve()}")

Saved 2,429 real public directory records to C:\Users\dhair\OneDrive\Desktop\College\Project\project implementation part\BloodBridge_AI\data\bloodbank_reference_catalog.csv
